In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import json
import joblib
import re
from openai import OpenAI

# =============================================================================
# 1. CONFIGURACIÓN E INICIALIZACIÓN DE LA PLATAFORMA
# =============================================================================
st.set_page_config(
    page_title="ClaraLap - Tu Asesor de Laptops",
    page_icon="💻",
    layout="wide"
)

# Clave de API de OpenAI (Puede ingresarse por UI o secrets de Streamlit)
OPENAI_API_KEY = st.sidebar.text_input("OpenAI API Key", type="password")

@st.cache_data
def cargar_catalogo():
    """
    Carga el catálogo de laptops procesado previamente.
    Si no existe el archivo procesado, simula la carga del CSV base.
    """
    try:
        df = pd.read_csv('catalogo_enriquecido.csv')
    except FileNotFoundError:
        st.error("No se encontró 'catalogo_enriquecido.csv'. Asegúrate de haber generado la base de datos.")
        st.stop()
    return df

@st.cache_resource
def cargar_modelo():
    """Carga el modelo de regresión entrenado y las columnas de entrada."""
    try:
        modelo = joblib.load('modelo_rf_laptops_precio.pkl')
        columnas = joblib.load('columnas_modelo.pkl')
        return modelo, columnas
    except FileNotFoundError:
        return None, None

df_catalogo = cargar_catalogo()
modelo_regresion, columnas_modelo = cargar_modelo()

# =============================================================================
# 2. FUNCIONES DEL AGENTE OPENAI (LLM)
# =============================================================================

def agente_extraer_requerimientos(prompt_usuario, client):
    """
    Agente 1: Convierte el texto libre del usuario en un JSON estructurado.
    """
    system_prompt = """
    Eres un asistente experto en traducir necesidades de usuarios a requerimientos de hardware de laptops.
    Debes responder ÚNICAMENTE con un objeto JSON válido con la siguiente estructura:
    {
      "presupuesto_max": float (si no se especifica, asigna 999999),
      "necesita_gpu_dedicada": bool,
      "gama_tecnica_minima": "Baja" | "Media" | "Alta",
      "ram_minima_gb": int (4, 8, 16 o 32),
      "portabilidad_prioritaria": bool,
      "resumen_uso": "string breve describiendo el caso de uso"
    }
    
    Reglas de interpretación:
    - Uso de software como AutoCAD, Blender, Premiere, Arquitectura, Renderizado, Gaming o Inteligencia Artificial requiere "necesita_gpu_dedicada": true, "gama_tecnica_minima": "Alta" o "Media", y "ram_minima_gb": 16.
    - Uso para Oficina, Excel pesado, Programación web o Universidad requiere "gama_tecnica_minima": "Media" y "ram_minima_gb": 8 u 16.
    - Uso básico, tareas escolares ligeras o navegación requiere "gama_tecnica_minima": "Baja" y "ram_minima_gb": 4 u 8.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt_usuario}
            ],
            response_format={"type": "json_object"},
            temperature=0.1
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        st.error(f"Error al procesar la consulta con OpenAI: {str(e)}")
        return None


def agente_traductor_empatico(perfil_usuario, laptops_recomendadas, client):
    """
    Agente 2: Genera explicaciones claras, empáticas y sin jerga técnica para el Top N.
    """
    system_prompt = """
    Eres el asesor virtual empático de ClaraLap. Tu objetivo es explicarle al usuario por qué
    las laptops seleccionadas por el algoritmo son perfectas para sus actividades diarias.
    
    Reglas:
    - Habla de manera cercana, amigable y profesional.
    - NUNCA uses jerga técnica abrumadora sin explicar su beneficio práctico.
    - Menciona explícitamente el 'Ahorro / Valor Justo' detectado por el modelo financiero.
    - Resalta aspectos concretos de su uso (ej. "No se trabará al renderizar", "Es muy ligera para llevar al trabajo").
    """

    user_content = f"""
    Perfil del usuario: {json.dumps(perfil_usuario, ensure_ascii=False)}
    Laptops recomendadas por el algoritmo:
    {laptops_recomendadas.to_json(orient='records', force_ascii=False)}
    
    Genera una recomendación estructurada para cada una de las laptops recomendadas.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_content}
            ],
            temperature=0.7
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error al generar la explicación: {str(e)}"

# =============================================================================
# 3. MOTOR DE MATCH MATEMÁTICO (FILTRADO Y RANKING)
# =============================================================================

def ejecutar_match_matematico(df, reqs):
    """
    Filtra y ordena el catálogo según los requerimientos JSON y el ahorro residual.
    """
    df_filtrado = df.copy()

    # 1. Filtro estricto por Presupuesto
    if reqs.get('presupuesto_max'):
        df_filtrado = df_filtrado[df_filtrado['Precio_MXN'] <= reqs['presupuesto_max']]

    # 2. Filtro por GPU Dedicada si se requiere
    if reqs.get('necesita_gpu_dedicada'):
        if 'GPU_Dedicada' in df_filtrado.columns:
            df_filtrado = df_filtrado[df_filtrado['GPU_Dedicada'] == 1]

    # 3. Filtro por RAM Mínima
    if reqs.get('ram_minima_gb'):
        df_filtrado = df_filtrado[df_filtrado['RAM_GB'] >= reqs['ram_minima_gb']]

    # 4. Filtro por Gama Técnica Determinista
    mapa_gamas = {'Baja': ['Baja', 'Media', 'Alta'], 'Media': ['Media', 'Alta'], 'Alta': ['Alta']}
    gamas_permitidas = mapa_gamas.get(reqs.get('gama_tecnica_minima', 'Baja'), ['Baja', 'Media', 'Alta'])
    
    if 'Gama_Tecnica' in df_filtrado.columns:
        df_filtrado = df_filtrado[df_filtrado['Gama_Tecnica'].isin(gamas_permitidas)]

    if df_filtrado.empty:
        return pd.DataFrame()

    # 5. Ordenamiento por Oportunidad / Residual (Mayor descuento respecto al valor justo)
    # Si la columna Diferencia_MXN ya existe, se usa directamente
    if 'Diferencia_MXN' in df_filtrado.columns:
        df_filtrado = df_filtrado.sort_values(by='Diferencia_MXN', ascending=True)
    else:
        df_filtrado = df_filtrado.sort_values(by='Precio_MXN', ascending=True)

    return df_filtrado.head(3)

# =============================================================================
# 4. INTERFAZ DE USUARIO (STREAMLIT)
# =============================================================================

st.title("💻 ClaraLap")
st.subheader("Tu Asesor Inteligente para Encontrar la Laptop Ideal")

st.markdown("""
Escribe en tus propias palabras qué actividades realizarás, qué software necesitas ejecutar o cuál es tu presupuesto. 
**Nuestra IA se encarga de traducir tus necesidades al hardware perfecto.**
""")

# Input de lenguaje natural
consulta_usuario = st.text_area(
    "¿Para qué vas a usar tu laptop?",
    placeholder="Ejemplo: Soy estudiante de arquitectura, necesito usar AutoCAD y Photoshop, viajo seguido a la universidad y mi presupuesto máximo es de $22,000 MXN.",
    height=120
)

col1, col2 = st.columns([1, 4])
with col1:
    btn_buscar = st.button("🔍 Encontrar mi Laptop", type="primary")

if btn_buscar:
    if not OPENAI_API_KEY:
        st.warning("Por favor ingresa tu API Key de OpenAI en la barra lateral para continuar.")
    elif not consulta_usuario.strip():
        st.warning("Por favor escribe tus requerimientos antes de realizar la búsqueda.")
    else:
        client = OpenAI(api_key=OPENAI_API_KEY)
        
        with st.spinner("Analizando tus requerimientos con IA..."):
            # Paso 1: Extracción JSON
            reqs = agente_extraer_requerimientos(consulta_usuario, client)
            
        if reqs:
            st.success(f"**Uso identificado:** {reqs.get('resumen_uso', 'General')}")
            
            with st.expander("Ver interpretación técnica del algoritmo (JSON)"):
                st.json(reqs)

            # Paso 2: Match Matemático en la Base de Datos
            top_laptops = ejecutar_match_matematico(df_catalogo, reqs)

            if top_laptops.empty:
                st.error("No encontramos laptops en el inventario que cumplan estrictamente con todos tus filtros y presupuesto. Intenta flexibilizar el presupuesto o las características.")
            else:
                st.markdown("---")
                st.subheader("🎯 Tus Mejores Opciones Seleccionadas")

                # Paso 3: Traducción Empática
                with st.spinner("Generando explicación personalizada sin jerga técnica..."):
                    explicacion = agente_traductor_empatico(reqs, top_laptops, client)

                st.markdown(explicacion)

                # Despliegue en tarjetas
                cols_cards = st.columns(len(top_laptops))
                for idx, (_, laptop) in enumerate(top_laptops.iterrows()):
                    with cols_cards[idx]:
                        st.image("https://via.placeholder.com/300x200?text=Laptop", use_container_width=True)
                        st.markdown(f"### {laptop['Marca']} - {laptop.get('Nombre', 'Laptop')[:30]}...")
                        st.markdown(f"**Precio:** ${laptop['Precio_MXN']:,.2f} MXN")
                        
                        if 'Precio_Justo_Modelo' in laptop:
                            precio_justo = laptop['Precio_Justo_Modelo']
                            ahorro = precio_justo - laptop['Precio_MXN']
                            if ahorro > 0:
                                st.success(f"🔥 ¡Ahorro estimado: ${ahorro:,.2f} MXN!")
                            else:
                                st.info(f"Valor Justo de Mercado: ${precio_justo:,.2f} MXN")

                        st.markdown(f"- **RAM:** {laptop.get('RAM_GB', 'N/A')} GB")
                        st.markdown(f"- **Almacenamiento:** {laptop.get('Almacenamiento_GB', 'N/A')} GB")
                        st.markdown(f"- **Gama Técnica:** {laptop.get('Gama_Tecnica', 'N/A')}")
                        st.button(f"Ver Oferta #{idx+1}", key=f"btn_{idx}")